In [10]:
import pandas as pd
import numpy as np
import re
DATASET = pd.read_csv('Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,NaN,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,NaN,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,NaN,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,NaN,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,NaN,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN


Consistency

In [11]:
# 1. Convert column to appropriate data type and handle NaN
# Now convert to numeric
DATASET['Civico'] = pd.to_numeric(DATASET['Civico'], errors='coerce')
DATASET['Civico'] = DATASET['Civico'].astype(pd.Int64Dtype())
DATASET['ZD'] = pd.to_numeric(DATASET['ZD'], errors='coerce')
DATASET['ZD'] = DATASET['ZD'].astype(pd.Int64Dtype())
DATASET['Superficie lavorativa'] = pd.to_numeric(DATASET['Superficie lavorativa'], errors='coerce')
DATASET['Superficie altri usi'] = pd.to_numeric(DATASET['Superficie altri usi'], errors='coerce')

# Ensure non-empty strings
DATASET['Tipo esercizio pa'] = DATASET['Tipo esercizio pa'].fillna("")

DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN


# Data wrangling

## Colonna Tipo esercizio pa

In [12]:
DATASET.rename(columns={'Prevalente':'Attività primaria','ZD':'Municipio'}, inplace=True)

In [13]:
#Standardization for "Tipo esercizio pa"
replacements = {
    'TIPO A - REG.2003': 'Estetista',
    'TIPO A ESTETICA MANUALE': 'Estetista',
    'TIPO B CENTRO DI ABBRONZATURA': 'Centro abbronzatura',
    'TIPO C TRATT.ESTETICI DIMAGRIM': 'Trattamento estetico dimagrimento',
    'TIPO D ESTET.APPAR.ELETTROMECC': 'Estetica con apparecchiature',
    'TIPO A-B-C-D': 'Estetista;Centro abbronzatura;Trattamento estetico dimagrimento;Estetica con apparecchiature',
    'TIPO A-B-C-D in profumeria': 'Estetista;Centro abbronzatura;Trattamento estetico dimagrimento;Estetica con apparecchiature',
    'BARBIERE': 'Barbiere',
    'ACCONCIATORE': 'Acconciatore',
    'esecuzione di tatuaggi e piercing': 'Esecuzione di tatuaggi e piercing'
}
for old_value, new_value in replacements.items():
    DATASET['Tipo esercizio pa'] = DATASET['Tipo esercizio pa'].str.replace(old_value, new_value, regex=False)

# Create boolean column for each category
categories = [
    'Estetista',
    'Centro abbronzatura',
    'Trattamento estetico dimagrimento',
    'Estetica con apparecchiature',       #TODO: Da capire se si può unire con Estetista
    'Barbiere',
    'Acconciatore',
    'Esecuzione di tatuaggi e piercing',
    'Truccatore'
]

for category in categories:
    DATASET[category] = DATASET['Tipo esercizio pa'].str.contains(category, na=False, regex=False)

DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,Estetista,Centro abbronzatura,Trattamento estetico dimagrimento,Estetica con apparecchiature,Barbiere,Acconciatore,Esecuzione di tatuaggi e piercing,Truccatore
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0,False,False,False,False,False,False,False,False
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0,False,False,False,False,False,False,False,False
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0,False,False,False,False,False,False,False,False
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN,False,False,False,False,False,False,False,False
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3904,Estetica con apparecchiature;Trattamento estet...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN,False,False,True,True,False,False,False,False
3905,Estetica con apparecchiature;Trattamento estet...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0,False,True,True,True,False,False,False,False
3906,Estetica con apparecchiature;Trattamento estet...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN,False,True,True,True,False,False,False,False
3907,Estetica con apparecchiature;Trattamento estet...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,False,True,True,True,False,False,False,False


In [ ]:
# Verifica valori unici
DATASET['Tipo esercizio pa'].str.split(';').explode().value_counts()

Tipo esercizio pa
Estetista                            1245
Parrucchiere per signora             1214
Acconciatore                          907
Centro abbronzatura                   693
Parrucchiere per uomo                 518
Trattamento estetico dimagrimento     226
Estetica con apparecchiature          162
Centro massaggi                       139
Parrucchiere misto                    105
                                       31
Centro benessere                       29
Esecuzione di tatuaggi e piercing      24
Pedicure estetico                      22
Manicure                                3
Barbiere                                2
Estetista in profumeria                 2
 (z.d. 9)                               1
Truccatore                              1
Name: count, dtype: int64

## Ubicazione

In [15]:
#Replacement for Ubicazione
DATASET['Ubicazione'] = DATASET['Ubicazione'].str.replace('num', 'N')

In [16]:
# Splitting the "Ubicazione" column on "N." to separate address and addition data
split_column = DATASET['Ubicazione'].str.split(r"\bN\.\s*", n=1, expand=True)
split_column.columns = ['Ubicazione_effettiva', 'Ubicazione_data']

# Add split columns to DATASET
DATASET = pd.concat([DATASET, split_column], axis=1)

# Splitting the "Ubicazione_data" column on "(" to separate civico and z.d.
split_column_2 = DATASET['Ubicazione_data'].str.split("(", n=1, expand=True)
split_column_2.columns = ['Civico_to_check', 'z.d._to_check']

# Add to original database
DATASET = pd.concat([DATASET, split_column_2], axis=1)

# Remove ";" from "Civico_to_check"
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].str.replace(";", "", regex=False)

# Remove the closing parenthesis ")" and the z.d. description from the "z.d._to_check" column
DATASET["z.d._to_check"] = DATASET["z.d._to_check"].str.replace("z.d.", "", regex=False)
DATASET["z.d._to_check"] = DATASET["z.d._to_check"].str.replace(")", "", regex=False)

# Splitting the "Ubicazione_effettiva" column on the first " " to separate "Tipo via" and "Via"
split_column_3 = DATASET['Ubicazione_effettiva'].str.split(" ", n=1, expand=True)
split_column_3.columns = ['Tipo via_to_check', 'Via_to_check']

# Add to original database
DATASET = pd.concat([DATASET, split_column_3], axis=1)

DATASET = DATASET.drop(['Ubicazione_effettiva', 'Ubicazione_data'], axis=1)
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,...,Trattamento estetico dimagrimento,Estetica con apparecchiature,Barbiere,Acconciatore,Esecuzione di tatuaggi e piercing,Truccatore,Civico_to_check,z.d._to_check,Tipo via_to_check,Via_to_check
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0,...,False,False,False,False,False,False,10,6,LGO,DEI GELSOMINI
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0,...,False,False,False,False,False,False,3,9,PZA,FIDIA
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0,...,False,False,False,False,False,False,10,5,VIA,ADIGE
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN,...,False,False,False,False,False,False,9,1,VIA,BARACCHINI FLAVIO
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0,...,False,False,False,False,False,False,12,4,VIA,BERGAMO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3904,Estetica con apparecchiature;Trattamento estet...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN,...,True,True,False,False,False,False,1 con ingr.da v.le montello 4/6,1,VIA,SARPI FRA' PAOLO
3905,Estetica con apparecchiature;Trattamento estet...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0,...,True,True,False,False,False,False,4,1,CSO,DI PORTA TICINESE
3906,Estetica con apparecchiature;Trattamento estet...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN,...,True,True,False,False,False,False,2,9,VIA,CANDOGLIA
3907,Estetica con apparecchiature;Trattamento estet...,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,...,True,True,False,False,False,False,002a,1,VIA,NIRONE


In [17]:
# Clean and convert Civico_to_check and z.d._to_check before comparison
def clean_numeric_string(value):
    if pd.isna(value):
        return pd.NA
    value_str = str(value).strip()
    if not value_str:
        return pd.NA
    # Split at first space and take only the numeric part
    value_str = value_str.split(' ')[0]
    # Remove leading zeros but keep at least one digit
    value_str = re.sub(r'^0+(?=\d)', '', value_str)
    if not value_str:
        value_str = "0"
    return value_str

# Apply cleaning to both columns
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].apply(clean_numeric_string)
DATASET['z.d._to_check'] = DATASET['z.d._to_check'].apply(clean_numeric_string)

# Now convert to Int64
DATASET['Civico_to_check'] = pd.to_numeric(DATASET['Civico_to_check'], errors='coerce').astype(pd.Int64Dtype())
DATASET['z.d._to_check'] = pd.to_numeric(DATASET['z.d._to_check'], errors='coerce').astype(pd.Int64Dtype())

# Check if the results are the same
condition = (DATASET['Civico_to_check'] == DATASET['Civico']) & (DATASET['z.d._to_check'] == DATASET['Municipio'])

# Show rows where condition is satisfied
DATASET[~condition]

#TODO: Manca il tipo via e il nome via, ma bisogna modificare un po' de robba (Non so se da fare)

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,...,Trattamento estetico dimagrimento,Estetica con apparecchiature,Barbiere,Acconciatore,Esecuzione di tatuaggi e piercing,Truccatore,Civico_to_check,z.d._to_check,Tipo via_to_check,Via_to_check
210,Acconciatore,codvia 4386 N.008; (z.d. 4),CSO,LODI,72,4068.0,4,NaN,NaN,50.0,...,False,False,False,True,False,False,8,4,codvia,4386
243,Acconciatore,mm1 duomo codvia 9112 N.000; (z.d. 1),CSO,SEMPIONE,10,7137.0,1,NaN,NaN,NaN,...,False,False,False,True,False,False,0,1,mm1,duomo codvia 9112
280,Acconciatore,VIA ALTAMURA SAVERIO N. 7 ; (z.d. 5),VIA,ALTAMURA SAVERIO,7,6576.0,7,NaN,NaN,NaN,...,False,False,False,True,False,False,7,5,VIA,ALTAMURA SAVERIO
368,Acconciatore,VIA CONSOLE MARCELLO N.0181; (z.d. 8),VIA,SILVA GUGLIELMO,49,6400.0,8,NaN,NaN,52.0,...,False,False,False,True,False,False,181,8,VIA,CONSOLE MARCELLO
429,Acconciatore,VIA FIUGGI N.0121; (z.d. 9),VIA,DE MARTINO EMILIO,1,1693.0,9,NaN,NaN,NaN,...,False,False,False,True,False,False,121,9,VIA,FIUGGI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3534,Estetista;Centro abbronzatura,CSO MAGENTA interno piano rialzato N.083; (z.d...,VIA,S. GIOV. SUL MURO,14,609.0,1,NaN,NaN,NaN,...,False,False,False,False,False,False,83,1,CSO,MAGENTA interno piano rialzato
3568,Estetista;Centro abbronzatura,VIA AGNELLO con ingr. in via san paolo n.7 N.0...,VIA,VOLTA ALESSANDRO,8,1018.0,1,NaN,NaN,NaN,...,False,False,False,False,False,False,61,1,VIA,AGNELLO con ingr. in via san paolo n.7
3699,Estetista;Centro abbronzatura,VIA PAGANO MARIO piano rialzato N.03739; (z.d. 1),CSO,VENEZIA,14,238.0,1,NaN,NaN,NaN,...,False,False,False,False,False,False,3739,1,VIA,PAGANO MARIO piano rialzato
3764,Estetista;Centro abbronzatura,VIA SERLIO SEBASTIANO N.0082; (z.d. 4),VIA,MINCIO,3,4157.0,4,NaN,NaN,NaN,...,False,False,False,False,False,False,82,4,VIA,SERLIO SEBASTIANO
